## CS310 Natural Language Processing
## Lab 6: Key-Value Cache

In this lab, we will practice the Key-Value (KV) cache techniqe. The code for this lab is adopted from the [LLMs-from-scratch (ch 04)](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch04/03_kv-cache)


In [ ]:
import tiktoken
import time
import torch
import torch.nn as nn

from utils import LayerNorm, GELU, FeedForward

### T1. Modify `MultiHeadAttention` and `TransformerBlock`

`MultiHeadAttention` is the first module that we need to add something new to.

The new class members are:
- `cache_k` and `cache_v`: buffers to store the cached keys and values
- `ptr_current_pos`: a pointer to the current position in the cache. Each time the `foward()` is called with `use_cache=True`, the `ptr_current_pos` needs be incremented by the input sequence length.

Note that in the previous implementation, `mask` is already registered as a buffer.

There are **three** places within the `forward()` method that you need to modify, all of which occur if `use_cache=True`:
- The `cache_k` and `cache_v` need to be updated by calling `torch.cat()`.
- The `mask_bool` needs to be truncated to only cover the current input length.
- The `ptr_current_pos` needs to be incremented by the input length.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1),
            persistent=False
        )

        ####################################################
        # NEW
        self.register_buffer("cache_k", None, persistent=False)
        self.register_buffer("cache_v", None, persistent=False)
        self.ptr_current_pos = 0
        ####################################################

    def forward(self, x, use_cache=False):
        b, num_tokens, d_in = x.shape

        keys_new = self.W_key(x)  # Shape: (b, num_tokens, d_out)
        values_new = self.W_value(x)
        queries = self.W_query(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys_new = keys_new.view(b, num_tokens, self.num_heads, self.head_dim)
        values_new = values_new.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        ####################################################
        # NEW
        # START YOUR CODE HERE
        if use_cache:
            if self.cache_k is None:
                # initialize cache
                pass
            else:
                # update cache by calling torch.cat()
                pass
            # retrieve cache
            keys, values = None, None
        else:
            keys, values = keys_new, values_new
        # END YOUR CODE HERE
        ####################################################

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        ####################################################
        # NEW
        # START YOUR CODE HERE
        num_tokens_Q = queries.shape[-2]
        num_tokens_K = keys.shape[-2]
        if use_cache:
            # compute mask
            mask_bool = None
            # update ptr_current_pos
            pass
        # END YOUR CODE HERE
        ####################################################
        # Original mask truncated to the number of tokens and converted to boolean
        else:
            mask_bool = self.mask.bool()[:num_tokens_Q, :num_tokens_K]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # optional projection

        return context_vec

    ####################################################
    # NEW
    def reset_cache(self):
        self.cache_k, self.cache_v = None, None
        self.ptr_current_pos = 0
    ####################################################

In [ ]:
# Test MultiHeadAttention 
torch.manual_seed(42)
d_in = 8
d_out = 8
context_length = 10
num_heads = 2
dropout = 0.0

mha = MultiHeadAttention(d_in, d_out, context_length, dropout, num_heads, qkv_bias=False)
mha.eval()

# Test input: batch_size=1, seq_len=3, d_in=8
x1 = torch.randn(1, 3, d_in)
output1 = mha(x1, use_cache=False)

# Reset cache
mha.reset_cache()

# Step 1: Process first 2 tokens
x_step1 = x1[:, :2, :]
output_step1 = mha(x_step1, use_cache=True)
# Step 2: Process next 1 token
x_step2 = x1[:, 2:3, :]
output_step2 = mha(x_step2, use_cache=True)
# Concatenate outputs
output_cached = torch.cat([output_step1, output_step2], dim=1)

print(f"\nNon-cached output shape: {output1.shape}")
print(f"Cached output shape: {output_cached.shape}")

# Check if outputs match (within numerical precision)
diff = torch.abs(output1 - output_cached).max().item()
print(f"\nMax absolute difference: {diff:.6f}")
print(f"Outputs match: {diff < 1e-5}")

print("\n✓ All tests passed!")

Next, `TransformerBlock` only needs a minor change to support calling the `MultiHeadAttention` module with `use_cache=True`.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x, use_cache=False):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)

        # x = self.att(x)   # Shape [batch_size, num_tokens, emb_size]
        ####################################################
        # NEW
        # START YOUR CODE HERE
        # call self.att properly
        x = None
        # END YOUR CODE HERE
        ####################################################

        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x

### T2. Modify `GPTModel`

There are several places in the `GPTModel` class that need to be modified:
- Add a `current_pos` attribute, whose value needs to be incremented by the input length when `use_cache=True`.
- Add a `use_cache` argument to `foward()`; When transfomer blocks are called, pass the `use_cache` argument to them.
- Change the way position embeddings are computed. The first token in the input should have the position ID of `self.current_pos`, the second token ID should be `self.current_pos + 1`, and so on.

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # self.trf_blocks = nn.Sequential(
        #    *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        ####################################################
        # NEW
        self.trf_blocks = nn.ModuleList(
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.current_pos = 0
        ####################################################

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx, use_cache=False):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)

        # pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))

        ####################################################
        # NEW
        # START YOUR CODE HERE
        if use_cache:
            # compute pos_ids
            pos_ids = None
            # update current_pos
            pass
        # END YOUR CODE HERE
        else:
            pos_ids = torch.arange(0, seq_len, device=in_idx.device, dtype=torch.long)
        pos_embeds = self.pos_emb(pos_ids).unsqueeze(0)
        ####################################################

        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)

        # x = self.trf_blocks(x)
        ####################################################
        # NEW
        for blk in self.trf_blocks:
            x = blk(x, use_cache=use_cache)
        ####################################################

        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

    ####################################################
    # NEW
    def reset_kv_cache(self):
        for blk in self.trf_blocks:
            blk.att.reset_cache()
        self.current_pos = 0
    ####################################################

### T3. Compare generation speed

Now it is time to see how the KV cache improves the generation speed.

First, we need to implement the generation loop that supports both modes: `use_cache=True` and `use_cache=False`.

Note:
- The size of input batch `idx` may exceed the context length. Thus, a limited slice of input should be fed to the model when computing `logits`. 
- Pick the token with the highest log-probability (greedy sampling) as `next_idx`.

In [ ]:
def generate_text_simple_cached(model, idx, max_new_tokens,
                                context_size=None, use_cache=True):
    model.eval()
    ctx_len = context_size or model.pos_emb.num_embeddings

    with torch.no_grad():
        if use_cache:
            # Init cache with full prompt
            model.reset_kv_cache()
            ### START YOUR CODE HERE ###
            # compute logits
            logits = None

            for _ in range(max_new_tokens):
                # a) pick the token with the highest log-probability (greedy sampling), using torch.argmax()
                next_idx = None
                # b) append it to the running sequence idx, by calling torch.cat()
                idx = None
                # c) recompute logits for next step by feeding model only the new token
                logits = None
            ### END YOUR CODE HERE ###
        else:
            ### START YOUR CODE HERE ###
            for _ in range(max_new_tokens):
                logits = None
                next_idx = None
                idx = None
            ### END YOUR CODE HERE ###

    return idx

Let's compare their speeds using some examples.

In [18]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "emb_dim": 768,          # Embedding dimension
    "n_heads": 12,           # Number of attention heads
    "n_layers": 12,          # Number of layers
    "drop_rate": 0.1,        # Dropout rate
    "qkv_bias": False        # Query-Key-Value bias
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()  # disable dropout
tokenizer = tiktoken.get_encoding("gpt2")

def test_generation_speed(use_cache=True, verbose=True):
    start_context = "Hello, I am"

    encoded = tokenizer.encode(start_context)
    encoded_tensor = torch.tensor(encoded, device=device).unsqueeze(0)

    if verbose:
        print(f"\n{50*'='}\n{22*' '}IN\n{50*'='}")
        print("\nInput text:", start_context)
        print("Encoded input text:", encoded)
        print("encoded_tensor.shape:", encoded_tensor.shape)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.time()

    token_ids = generate_text_simple_cached(
        model=model,
        idx=encoded_tensor,
        max_new_tokens=200,
        use_cache=use_cache
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    total_time = time.time() - start

    decoded_text = tokenizer.decode(token_ids.squeeze(0).tolist())

    if verbose:
        print(f"\n\n{50*'='}\n{22*' '}OUT\n{50*'='}")
        print("\nOutput:", token_ids)
        print("Output length:", len(token_ids[0]))
        print("Output text:", decoded_text)

    print(f"\nMode: use_cache={use_cache}")
    print(f"Time: {total_time:.2f} sec")
    print(f"{int(len(token_ids[0])/total_time)} tokens/sec")
    if torch.cuda.is_available():
        max_mem_bytes = torch.cuda.max_memory_allocated()
        max_mem_gb = max_mem_bytes / (1024 ** 3)
        print(f"Max memory allocated: {max_mem_gb:.2f} GB")

In [19]:
test_generation_speed(use_cache=True, verbose=False)
test_generation_speed(use_cache=False, verbose=False)


Mode: use_cache=True
Time: 3.22 sec
63 tokens/sec

Mode: use_cache=False
Time: 12.91 sec
15 tokens/sec
